# Cheat Sheet: Survival Analysis with `lifelines` / `statsmodels`

A fast reference for Kaplan-Meier, Exponential/parametric, and Cox Proportional Hazards models.
Every snippet below is copy-paste ready — swap in your own `duration_col` / `event_col` / covariates.

**Install:** `pip install lifelines==0.27.4 statsmodels pandas matplotlib --break-system-packages`


## Imports (the whole toolkit in one cell)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from lifelines import (KaplanMeierFitter, NelsonAalenFitter, ExponentialFitter,
                        WeibullFitter, LogNormalFitter, LogLogisticFitter, CoxPHFitter)
from lifelines.statistics import logrank_test, multivariate_logrank_test, proportional_hazard_test
from lifelines.utils import concordance_index, median_survival_times
import statsmodels.api as sm


## Core vocabulary

| Symbol | Name | Meaning |
|---|---|---|
| `duration` | Y | observed time = min(true event time, censoring time) |
| `event_observed` | δ | 1 if the event happened, 0 if censored |
| S(t) | Survival function | P(T > t) |
| h(t) | Hazard function | instantaneous event rate given survival to t |
| H(t) | Cumulative hazard | ∫h(t) dt; S(t) = exp(−H(t)) |
| HR | Hazard ratio | ratio of hazards between two groups/covariate levels |

Right-censoring is the default assumption of every fitter below unless stated otherwise.


## 1. Kaplan-Meier (non-parametric)

In [ ]:
kmf = KaplanMeierFitter()
kmf.fit(durations=df['time'], event_observed=df['event'], label='cohort')

kmf.survival_function_       # table of S(t) at each observed time
kmf.confidence_interval_survival_function_   # 95% CI band, as a DataFrame
kmf.median_survival_time_    # single number
kmf.predict(30)              # S(t) at t = 30
kmf.event_table              # at-risk / observed / censored counts per time step


In [ ]:
# Plot with CI band
fig, ax = plt.subplots()
kmf.plot_survival_function(ax=ax)          # shorthand for .plot()
# manual CI shading if you want more control:
ci = kmf.confidence_interval_survival_function_
ax.fill_between(ci.index, ci.values[:, 0], ci.values[:, 1], alpha=0.3, color='gray')


In [ ]:
# Compare two or more groups
kmf_a = KaplanMeierFitter().fit(df.loc[df.group=='A','time'], df.loc[df.group=='A','event'], label='A')
kmf_b = KaplanMeierFitter().fit(df.loc[df.group=='B','time'], df.loc[df.group=='B','event'], label='B')
ax = kmf_a.plot()
kmf_b.plot(ax=ax)


In [ ]:
# Two-group log-rank test
result = logrank_test(durations_A=df.loc[df.group=='A','time'], durations_B=df.loc[df.group=='B','time'],
                       event_observed_A=df.loc[df.group=='A','event'], event_observed_B=df.loc[df.group=='B','event'])
result.print_summary()      # p-value, test statistic
result.p_value

# 3+ group log-rank test
mv = multivariate_logrank_test(df['time'], df['group'], df['event'])
mv.print_summary()


## 2. Parametric models (Exponential / Weibull / LogNormal / LogLogistic)

In [ ]:
exf = ExponentialFitter().fit(df['time'], df['event'], label='Exponential')
exf.lambda_                  # fitted rate parameter
exf.survival_function_
exf.plot_survival_function()
exf.AIC_                     # for model comparison — lower is better


In [ ]:
# Compare several parametric families by AIC in one pass
candidates = {'Exponential': ExponentialFitter(), 'Weibull': WeibullFitter(),
              'LogNormal': LogNormalFitter(), 'LogLogistic': LogLogisticFitter()}
aic = {}
for name, fitter in candidates.items():
    fitter.fit(df['time'], df['event'])
    aic[name] = fitter.AIC_
sorted(aic.items(), key=lambda kv: kv[1])   # best (lowest AIC) first


**When to reach for these:** Exponential assumes a *constant* hazard — rare in real data. Weibull allows hazard to rise or fall monotonically. LogNormal/LogLogistic allow non-monotonic (rise-then-fall) hazards. Always compare against Kaplan-Meier visually and by AIC before trusting a parametric shape.

## 3. Cox Proportional Hazards (semi-parametric, multivariate)

In [ ]:
cph = CoxPHFitter()
cph.fit(df=model_df, duration_col='time', event_col='event')   # model_df must contain ONLY duration, event, covariates
cph.print_summary()          # coef, exp(coef)=HR, CI, p-value, concordance, AIC, etc.


In [ ]:
# Reading the output
cph.summary                      # full table as a DataFrame
cph.summary['exp(coef)']         # hazard ratios
cph.hazard_ratios_                # same thing, convenience accessor
cph.concordance_index_           # discrimination: 0.5=random, 1.0=perfect ranking

cph.plot()                       # forest plot of log(HR) with 95% CI


In [ ]:
# Predict for new observations (dataframe with ONLY the covariate columns, same names/order as fit)
cph.predict_survival_function(new_df).plot(legend=False)
cph.predict_partial_hazard(new_df)     # relative risk score (higher = higher hazard)
cph.predict_median(new_df)             # predicted median survival time


In [ ]:
# Concordance index computed manually (sanity check against cph.concordance_index_)
c = concordance_index(model_df['time'], -cph.predict_partial_hazard(model_df), model_df['event'])


In [ ]:
# REQUIRED diagnostic: check the proportional hazards assumption before trusting the model
cph.check_assumptions(model_df, p_value_threshold=0.05, show_plots=True)
# If a covariate fails: consider stratification (cph.fit(..., strata=['col'])) or a time-varying covariate model


## 4. Interpretation cheat sheet


- **`exp(coef) = 1.00`** → no effect on hazard.
- **`exp(coef) = 1.30`** → a one-unit increase in that covariate multiplies the hazard by 1.30 (30% higher
  hazard), holding other covariates fixed.
- **`exp(coef) = 0.70`** → a one-unit increase *reduces* hazard by 30% (0.70 = 1 − 0.30).
- **For a 0/1 dummy** (e.g., `transplant`): `exp(coef)` compares the "1" group to the "0" group directly.
- A **higher hazard** = **shorter** expected survival. Don't mix up hazard direction with survival direction.
- `coef` is on the log-hazard scale; `exp(coef)` is the interpretable hazard ratio scale.
- A **significant p-value with a CI on `exp(coef)` that excludes 1** = statistically significant effect.


## 5. Common pitfalls


1. **Passing extra columns into `CoxPHFitter.fit()`.** Any column in the dataframe that isn't
   `duration_col`, `event_col`, or a real covariate will be treated as a covariate. Slice your dataframe
   first.
2. **Forgetting to check proportional hazards.** `print_summary()` will run even if the assumption is
   badly violated — it won't warn you. Always call `check_assumptions()`.
3. **Confusing `event_observed=1` with "died"/"failed" vs. "censored".** Convention: 1 = event happened,
   0 = censored. Double-check your raw data's coding before fitting.
4. **Reading Kaplan-Meier confidence bands as significance tests.** Overlapping bands are *suggestive*
   of no difference but are not a formal test — use `logrank_test` instead.
5. **Trusting AIC comparisons across models fit to different subsets of data.** AIC is only comparable
   across models fit to the *same* observations.
6. **Extrapolating past the last observed event time.** Kaplan-Meier and Cox baseline hazards are
   undefined beyond the last event in the data; parametric models will happily extrapolate but the
   result is a modeling assumption, not an observation.
7. **Small samples inflate uncertainty.** Both example datasets here (n=90, n=103) are small — treat
   p-values and confidence intervals as approximate, not exact.
